In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

# Device configuration (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# -------------------------------------------------------------
# 1. Download & Preprocess (Transforms handle /255.0 normalization)
# -------------------------------------------------------------
transform = transforms.ToTensor()

full_train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

# 80/20 Train/Validation Split
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

print(f"Loaded CIFAR-10 successfully! Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")

# -------------------------------------------------------------
# 2. Define Model Architecture
# -------------------------------------------------------------
class MLP(nn.Module):
    def __init__(self, hidden_units, activation):
        super(MLP, self).__init__()
        act_fn = nn.ReLU() if activation == 'relu' else nn.Tanh()
        
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 3, hidden_units[0]),
            act_fn,
            nn.Linear(hidden_units[0], hidden_units[1]),
            act_fn,
            nn.Linear(hidden_units[1], 10) # Logits output for CrossEntropyLoss
        )

    def forward(self, x):
        return self.model(x)

# -------------------------------------------------------------
# 3. Training & Evaluation Functions
# -------------------------------------------------------------
def train_epoch(model, dataloader, criterion, optimizer):
    model.train()
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

def evaluate(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

# -------------------------------------------------------------
# 4. Hyperparameter Search Loop
# -------------------------------------------------------------
best_acc = 0.0
best_model_state = None
best_config = None

hyperparameter_configs = [
    ([256, 128], 'relu'),
    ([128, 64], 'relu'),
    ([256, 128], 'tanh')
]

criterion = nn.CrossEntropyLoss()

for hidden_units, activation in hyperparameter_configs:
    print(f"\n--- Training Model with Units={hidden_units} | Activation='{activation}' ---")
    
    model = MLP(hidden_units, activation).to(device)
    optimizer = optim.Adam(model.parameters())
    
    for epoch in range(5):
        train_epoch(model, train_loader, criterion, optimizer)
        val_acc = evaluate(model, val_loader)
        print(f"Epoch {epoch+1}/5 - Validation Accuracy: {val_acc * 100:.2f}%")
        
    test_acc = evaluate(model, test_loader)
    print(f"Test Accuracy: {test_acc * 100:.2f}%")
    
    if test_acc > best_acc:
        best_acc = test_acc
        best_config = (hidden_units, activation)
        best_model_state = model.state_dict()

# -------------------------------------------------------------
# 5. Print Results & Sample Prediction
# -------------------------------------------------------------
print("\n" + "="*50)
print(f" Best Accuracy: {best_acc * 100:.2f}%")
print(f" Best Config  : Units={best_config[0]}, Activation='{best_config[1]}'")
print("="*50 + "\n")

# Load best model for inference
best_model = MLP(best_config[0], best_config[1]).to(device)
best_model.load_state_dict(best_model_state)
best_model.eval()

# Sample prediction on index 0 of test set
sample_image, true_label_idx = test_dataset[0]
input_tensor = sample_image.unsqueeze(0).to(device) # Add batch dimension: (1, 3, 32, 32)

with torch.no_grad():
    prediction_logits = best_model(input_tensor)
    predicted_label_idx = torch.argmax(prediction_logits, dim=1).item()

# Convert PyTorch tensor (C, H, W) to NumPy (H, W, C) for display
display_image = sample_image.permute(1, 2, 0).numpy()

plt.figure(figsize=(3, 3))
plt.imshow(display_image)
plt.axis('off')
plt.title(f"Predicted: {class_names[predicted_label_idx]}\nTrue: {class_names[true_label_idx]}")
plt.show()

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\Student\AppData\Roaming\Python\Python310\site-packages\torch\lib\c10.dll" or one of its dependencies.